In [ ]:
"""
# Object-Based Defect Detection Pipeline

This notebook demonstrates object-level defect detection pipeline using modular components.
"""

# Import all modules with correct paths
import sys
import os

# Add the abbvisionsystem directory to the Python path
sys.path.append('abbvisionsystem')
sys.path.append('abbvisionsystem/model_training')
sys.path.append('abbvisionsystem/models')

# Now import the modules
from abbvisionsystem.model_training.data_manager import DataManager, SyntheticDataGenerator
from abbvisionsystem.model_training.model_trainer import DefectClassificationTrainer, AnomalyDetectionTrainer
from abbvisionsystem.models.defect_detection_model import ObjectDefectDetectionModel
from abbvisionsystem.model_training.visualization import Visualizer
from abbvisionsystem.model_training.object_detector import ObjectDetector

import logging
import os

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration
CONFIG = {
    'source_dir': 'data/choco-pie',
    'object_dataset': 'object_detection_dataset',
    'model_name': 'object_defect_classifier',
    'epochs': 50,
    'batch_size': 32,
    'image_size': (224, 224)
}

print("🚀 Starting Object-Based Defect Detection Pipeline")
print("=" * 50)

# Step 1: Data Organization
print("\n📁 Step 1: Organizing Data for Object Detection")
data_manager = DataManager(CONFIG['source_dir'], CONFIG['object_dataset'])

# Organize for object detection - this now handles good, defect, AND both folders
print("Organizing for object-level detection...")
try:
    metadata = data_manager.organize_object_detection_data()
    print(f"✅ Object detection data organized successfully")
    print(f"📊 Created metadata with {len(metadata)} object entries")
except Exception as e:
    print(f"❌ Error organizing object detection data: {str(e)}")
    exit(1)

# Check what directories were created
object_dir = f"{CONFIG['object_dataset']}_objects"
print(f"\n📂 Object detection directory: {object_dir}")
if os.path.exists(object_dir):
    print(f"✅ Object detection directory exists")
    for split in ['train', 'validation', 'test']:
        split_dir = os.path.join(object_dir, split)
        if os.path.exists(split_dir):
            normal_count = len([f for f in os.listdir(os.path.join(split_dir, 'normal')) 
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            defect_count = len([f for f in os.listdir(os.path.join(split_dir, 'defect')) 
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            both_count = len([f for f in os.listdir(os.path.join(split_dir, 'both')) 
                            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]) if os.path.exists(os.path.join(split_dir, 'both')) else 0
            print(f"   {split}: {normal_count} normal, {defect_count} defect, {both_count} both objects")
else:
    print(f"❌ Object detection directory does not exist")
    exit(1)

# Step 2: Data Visualization
print("\n🎨 Step 2: Data Visualization")
visualizer = Visualizer()
try:
    visualizer.visualize_object_extraction(dataset_dir=object_dir)
except Exception as e:
    print(f"⚠️ Visualization error: {str(e)}")

# Step 3: Object Detection Model Training
print("\n🏋️ Step 3: Training Object Detection Model")

# Check if we have enough data
train_dir = os.path.join(object_dir, 'train')
if os.path.exists(train_dir):
    normal_files = [f for f in os.listdir(os.path.join(train_dir, 'normal')) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    defect_files = [f for f in os.listdir(os.path.join(train_dir, 'defect')) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    print(f"📊 Training data: {len(normal_files)} normal objects, {len(defect_files)} defect objects")
    
    if len(normal_files) > 0 or len(defect_files) > 0:
        try:
            trainer = DefectClassificationTrainer(
                data_dir=object_dir,
                model_name=CONFIG['model_name'],
                image_size=CONFIG['image_size']
            )
            
            model = trainer.create_model()
            print(f"Model created with {model.count_params():,} parameters")
            
            # Train model
            history = trainer.train(epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'])
            
            # Step 4: Model Evaluation
            print("\n📊 Step 4: Model Evaluation")
            results = trainer.evaluate()
            
            # Plot training history
            trainer.plot_training_history()
            
            # Step 5: Save Model
            print("\n💾 Step 5: Saving Model")
            trainer.save_model()
            
            print("✅ Object detection model trained and saved!")
            
        except Exception as e:
            print(f"❌ Error training model: {str(e)}")
            exit(1)
    else:
        print("⚠️ No training data found")
        exit(1)
else:
    print(f"❌ Training directory not found: {train_dir}")
    exit(1)

# Step 6: Test Object Detection Model
print("\n🧪 Step 6: Testing Object Detection Model")
object_detection_model = ObjectDefectDetectionModel(
    classifier_path=f"trained_models/{CONFIG['model_name']}.h5"
)

if object_detection_model.load():
    print("✅ Object detection model loaded successfully")
    
    # Test on validation images including 'both' category
    test_dirs = [
        f"{object_dir}/../test/normal",
        f"{object_dir}/../test/defect", 
        f"{object_dir}/../test/both"
    ]
    
    for test_dir in test_dirs:
        if os.path.exists(test_dir):
            category = os.path.basename(test_dir)
            print(f"\n🔍 Testing on {category} images:")
            
            test_files = [f for f in os.listdir(test_dir)[:3] 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            
            for img_file in test_files:
                try:
                    import cv2
                    img_path = os.path.join(test_dir, img_file)
                    img = cv2.imread(img_path)
                    if img is not None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        
                        detections = object_detection_model.predict(img)
                        print(f"   {img_file}: {detections['num_detections']} objects detected")
                        
                        if detections['num_detections'] > 0:
                            result_img = object_detection_model.visualize_detections(img, detections)
                            result_path = f"object_detection_result_{category}_{img_file}"
                            cv2.imwrite(result_path, cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))
                            print(f"   💾 Saved result to {result_path}")
                            
                            # Print detection details
                            for i, detail in enumerate(detections.get('object_details', [])):
                                class_id = detections['classes'][i]
                                confidence = detections['scores'][i]
                                class_name = object_detection_model.categories.get(class_id, {}).get('name', f'Class {class_id}')
                                print(f"      Object {i+1}: {class_name} ({confidence:.2%})")
                                
                except Exception as e:
                    print(f"⚠️ Error testing on {img_file}: {str(e)}")
else:
    print("❌ Failed to load object detection model")

# Step 7: Anomaly Detection (Optional)
print("\n🔍 Step 7: Anomaly Detection Training")
try:
    print("Training anomaly detection model...")
    
    normal_data_dir = f"{object_dir}/train/normal"
    if os.path.exists(normal_data_dir):
        anomaly_trainer = AnomalyDetectionTrainer(normal_data_dir=normal_data_dir)
        autoencoder = anomaly_trainer.create_autoencoder()
        anomaly_history = anomaly_trainer.train(epochs=30)
        anomaly_trainer.save_model()
        print("✅ Anomaly detection model trained and saved")
    else:
        print("⚠️ No normal training data found for anomaly detection")
        
except Exception as e:
    print(f"⚠️ Anomaly detection training skipped: {str(e)}")

# Final Summary
print("\n🎉 Pipeline Complete!")
print("=" * 50)
print("✅ Object detection data organized")
print("✅ Object detection model trained and saved")
print("✅ Model evaluation completed")
print("✅ Validation on all categories completed")

# Check what models were created
if os.path.exists("trained_models"):
    models = [f for f in os.listdir("trained_models") if f.endswith(('.h5', '.keras'))]
    print(f"✅ {len(models)} models saved in 'trained_models/' directory:")
    for model_file in models:
        print(f"   📄 {model_file}")
        
    if f"{CONFIG['model_name']}.h5" in models:
        print("🎯 Object detection model is ready for deployment!")
    else:
        print("⚠️ Object detection model not found")
else:
    print("❌ No trained_models directory found")

print("\n📋 Next Steps:")
print("1. Review object detection results")
print("2. Launch Streamlit app: streamlit run abbvisionsystem/app.py")
print("3. Test with new images")
print("4. Deploy to production environment")